In [ ]:
# notebooks/05_complete_backtest.ipynb
"""
Complete Task 5: Strategy Backtesting against 60/40 Benchmark
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TASK 5: STRATEGY BACKTESTING")
print("="*60)

# Load data
def load_backtest_data(tickers, start_date, end_date):
    """Load and prepare data for backtesting."""
    returns_df = pd.DataFrame()
    
    for ticker in tickers:
        try:
            df = pd.read_csv(f'../data/processed/{ticker}_data.csv', parse_dates=['Date'])
            df.set_index('Date', inplace=True)
            
            # Filter to backtest period
            df = df[(df.index >= start_date) & (df.index <= end_date)]
            
            if df.empty:
                raise ValueError(f"No data for {ticker} in backtest period")
            
            returns_df[ticker] = df['Close'].pct_change()
            
        except Exception as e:
            print(f"Error loading {ticker}: {e}")
            raise
    
    returns_df.dropna(inplace=True)
    
    if returns_df.empty:
        raise ValueError("No valid backtest data")
    
    print(f"✓ Backtest data: {len(returns_df)} rows ({returns_df.index.min()} to {returns_df.index.max()})")
    return returns_df

# Define backtest period (last year)
backtest_start = '2025-01-01'
backtest_end = '2026-06-30'

tickers = ['TSLA', 'BND', 'SPY']
backtest_returns = load_backtest_data(tickers, backtest_start, backtest_end)

# Define portfolio weights (from Task 4 - Max Sharpe)
strategy_weights = {
    'TSLA': 0.45,
    'BND': 0.25,
    'SPY': 0.30
}

benchmark_weights = {
    'TSLA': 0.0,
    'BND': 0.40,
    'SPY': 0.60
}

print(f"\nPortfolio Weights:")
print("  Strategy (from Task 4):")
for ticker, w in strategy_weights.items():
    print(f"    {ticker}: {w*100:.1f}%")
print("  Benchmark (60/40):")
for ticker, w in benchmark_weights.items():
    print(f"    {ticker}: {w*100:.1f}%")

# Calculate portfolio returns
def calculate_returns(returns_df, weights):
    """Calculate weighted portfolio returns with validation."""
    total_weight = sum(weights.values())
    if abs(total_weight - 1) > 1e-6:
        raise ValueError(f"Weights don't sum to 1: {total_weight}")
    
    portfolio_returns = pd.Series(0, index=returns_df.index)
    for ticker, weight in weights.items():
        if ticker not in returns_df.columns:
            raise ValueError(f"Ticker {ticker} not in returns data")
        portfolio_returns += returns_df[ticker] * weight
    
    return portfolio_returns

try:
    strategy_returns = calculate_returns(backtest_returns, strategy_weights)
    benchmark_returns = calculate_returns(backtest_returns, benchmark_weights)
    
    # Calculate cumulative returns
    strategy_cumulative = (1 + strategy_returns).cumprod()
    benchmark_cumulative = (1 + benchmark_returns).cumprod()
    
except Exception as e:
    print(f"Portfolio calculation failed: {e}")
    raise

# Calculate performance metrics
def calculate_metrics(returns, name):
    """Calculate comprehensive performance metrics."""
    if returns.empty:
        raise ValueError(f"Empty returns for {name}")
    
    # Total return
    total_return = (1 + returns).prod() - 1
    
    # Annualized return (252 trading days)
    ann_return = (1 + total_return) ** (252 / len(returns)) - 1
    
    # Annualized volatility
    ann_vol = returns.std() * np.sqrt(252)
    
    # Sharpe ratio (2% risk-free rate)
    risk_free_rate = 0.02
    sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else 0
    
    # Maximum drawdown
    cumulative = (1 + returns).cumprod()
    rolling_max = cumulative.expanding().max()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    return {
        'Total Return': total_return * 100,
        'Annualized Return': ann_return * 100,
        'Annualized Volatility': ann_vol * 100,
        'Sharpe Ratio': sharpe,
        'Maximum Drawdown': max_drawdown * 100
    }

strategy_metrics = calculate_metrics(strategy_returns, "Strategy")
benchmark_metrics = calculate_metrics(benchmark_returns, "Benchmark")

# Create comparison table
comparison_df = pd.DataFrame({
    'Strategy': strategy_metrics,
    'Benchmark': benchmark_metrics
}).round(2)

print("\n" + "="*60)
print("PERFORMANCE METRICS COMPARISON")
print("="*60)
print(comparison_df)

# Plot cumulative returns
fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(strategy_cumulative.index, strategy_cumulative, 
        label='Strategy Portfolio', color='blue', linewidth=2.5)
ax.plot(benchmark_cumulative.index, benchmark_cumulative, 
        label='Benchmark (60/40)', color='green', linewidth=2, linestyle='--')

ax.axhline(y=1, color='black', linestyle='-', alpha=0.3)
ax.set_title('Strategy vs Benchmark: Cumulative Returns', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Returns')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Conclusion
print("\n" + "="*60)
print("CONCLUSION")
print("="*60)

outperformed = strategy_metrics['Total Return'] > benchmark_metrics['Total Return']
print(f"\n{'✅' if outperformed else '❌'} The strategy {'OUTPERFORMED' if outperformed else 'UNDERPERFORMED'} the benchmark")

print(f"\nStrategy vs Benchmark:")
print(f"  Total Return: {strategy_metrics['Total Return']:.2f}% vs {benchmark_metrics['Total Return']:.2f}%")
print(f"  Sharpe Ratio: {strategy_metrics['Sharpe Ratio']:.3f} vs {benchmark_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown: {strategy_metrics['Maximum Drawdown']:.2f}% vs {benchmark_metrics['Maximum Drawdown']:.2f}%")

print("\nLimitations:")
print("  1. Limited to one year of data")
print("  2. Does not account for transaction costs")
print("  3. Assumes perfect rebalancing execution")

print("\n" + "="*60)
print("TASK 5 COMPLETE")
print("="*60)